In [ ]:
# Install dependencies (Colab)
!pip install -q lightfm-next numpy scipy scikit-learn

import json
import os
import csv
from datetime import datetime
import math
import numpy as np
from scipy import sparse
from lightfm import LightFM

def load_sparse_matrix(path: str) -> sparse.csr_matrix:
    matrix = sparse.load_npz(path)
    if not sparse.isspmatrix_csr(matrix):
        matrix = matrix.tocsr()
    return matrix

def _top_k_indices(scores: np.ndarray, k: int) -> np.ndarray:
    if k <= 0:
        return np.array([], dtype=np.int64)
    if k >= scores.shape[0]:
        return np.argsort(-scores)
    idx = np.argpartition(-scores, k - 1)[:k]
    return idx[np.argsort(-scores[idx])]

def compute_user_metrics(
    scores: np.ndarray,
    train_items: np.ndarray,
    test_items: np.ndarray,
    k: int,
    use_ndcg: bool = True,
) -> tuple[float, float, float] | None:
    if test_items.size == 0 or k <= 0 or scores.size == 0:
        return None
    scores = np.asarray(scores, dtype=float)
    scores = scores.copy()
    scores[train_items] = -np.inf
    k = min(k, scores.size)
    ranked = _top_k_indices(scores, k)
    test_set = set(test_items.tolist())
    hits = sum(1 for item in ranked if item in test_set)
    precision = hits / k
    recall = hits / len(test_set)
    ndcg = 0.0
    if use_ndcg:
        dcg = 0.0
        for rank, item in enumerate(ranked):
            if item in test_set:
                dcg += 1.0 / math.log2(rank + 2)
        ideal = min(len(test_set), k)
        idcg = sum(1.0 / math.log2(rank + 2) for rank in range(ideal))
        ndcg = (dcg / idcg) if idcg > 0 else 0.0
    return precision, recall, ndcg

def evaluate_ranking(
    score_fn,
    train_interactions: sparse.csr_matrix,
    test_interactions: sparse.csr_matrix,
    k: int,
    use_ndcg: bool = True,
) -> dict:
    precisions = []
    recalls = []
    ndcgs = []
    num_users = train_interactions.shape[0]
    for user_id in range(num_users):
        test_items = test_interactions[user_id].indices
        if test_items.size == 0:
            continue
        train_items = train_interactions[user_id].indices
        scores = score_fn(user_id)
        metrics = compute_user_metrics(scores, train_items, test_items, k, use_ndcg)
        if metrics is None:
            continue
        precision, recall, ndcg = metrics
        precisions.append(precision)
        recalls.append(recall)
        if use_ndcg:
            ndcgs.append(ndcg)
    result = {
        "k": float(k)
        ,"users_evaluated": float(len(precisions))
        ,"precision_at_k": float(np.mean(precisions)) if precisions else 0.0
        ,"recall_at_k": float(np.mean(recalls)) if recalls else 0.0
    }
    if use_ndcg:
        result["ndcg_at_k"] = float(np.mean(ndcgs)) if ndcgs else 0.0
    return result

def recommend_top_k(scores: np.ndarray, train_items: np.ndarray, k: int) -> np.ndarray:
    scores = np.asarray(scores, dtype=float)
    scores = scores.copy()
    scores[train_items] = -np.inf
    k = min(k, scores.size)
    return _top_k_indices(scores, k)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.3 MB/s eta 0:00:00


In [ ]:
# Config (match train_pure_cf.py defaults)
DATA_DIR_P5 = "phase5_outputs"
DATA_DIR_P4 = "phase4_outputs"
OUTPUT_DIR = os.path.join("outputs", f"pure_cf_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
K = 10
LOSS = "warp"
NO_COMPONENTS = 64
LEARNING_RATE = 0.05
EPOCHS = 20
RANDOM_STATE = 42
NUM_THREADS = 4
USE_NDCG = True
EXAMPLE_USERS = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)

train_interactions_csr = load_sparse_matrix(os.path.join(DATA_DIR_P5, "train_interactions.npz"))
test_interactions = load_sparse_matrix(os.path.join(DATA_DIR_P5, "test_interactions.npz"))
train_weights_csr = load_sparse_matrix(os.path.join(DATA_DIR_P5, "train_weights.npz"))

# LightFM training expects COO for interactions/weights.
train_interactions = train_interactions_csr.tocoo()
train_weights = train_weights_csr.tocoo()
test_interactions = test_interactions.tocsr()

print("Train shape:", train_interactions_csr.shape)
print("Test shape:", test_interactions.shape)
print("Train weights nnz:", train_weights.nnz)

Train shape: (79153, 20700)
Test shape: (79153, 20700)
Train weights nnz: 1539303


In [ ]:
# Train pure CF model (no item features)
model = LightFM(
    loss=LOSS,
    no_components=NO_COMPONENTS,
    learning_rate=LEARNING_RATE,
    random_state=RANDOM_STATE,
)

model.fit(
    train_interactions,
    sample_weight=train_weights,
    epochs=EPOCHS,
    num_threads=NUM_THREADS,
)

In [ ]:
# Evaluate + save outputs
num_items = train_interactions_csr.shape[1]
item_ids = np.arange(num_items, dtype=np.int64)

def score_fn(user_id: int) -> np.ndarray:
    return model.predict(user_id, item_ids)

metrics = evaluate_ranking(
    score_fn,
    train_interactions_csr,
    test_interactions,
    k=K,
    use_ndcg=USE_NDCG,
 )

print("Pure CF metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value}")

with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

with open(os.path.join(OUTPUT_DIR, "metrics.csv"), "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(metrics.keys()))
    writer.writeheader()
    writer.writerow(metrics)

# Recommendation examples (optional book/user maps)
book2id_path = os.path.join(DATA_DIR_P4, "book2id.json")
user2id_path = os.path.join(DATA_DIR_P4, "user2id.json")
id2book = {}
id2user = {}
if os.path.exists(book2id_path):
    with open(book2id_path, "r", encoding="utf-8") as f:
        id2book = {v: k for k, v in json.load(f).items()}
if os.path.exists(user2id_path):
    with open(user2id_path, "r", encoding="utf-8") as f:
        id2user = {v: k for k, v in json.load(f).items()}

rng = np.random.RandomState(RANDOM_STATE)
users_with_test = np.where(test_interactions.getnnz(axis=1) > 0)[0]
sampled_users = rng.choice(
    users_with_test,
    size=min(EXAMPLE_USERS, users_with_test.size),
    replace=False,
 ) if users_with_test.size > 0 else np.array([], dtype=int)

with open(os.path.join(OUTPUT_DIR, "recommendation_examples.csv"), "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["user_id", "user_index", "recommended_item_ids", "recommended_book_ids"])
    for user_id in sampled_users:
        train_items = train_interactions_csr[user_id].indices
        scores = score_fn(int(user_id))
        topk = recommend_top_k(scores, train_items, K)
        rec_items = [str(int(i)) for i in topk]
        rec_books = [id2book.get(int(i), "") for i in topk]
        writer.writerow([
            id2user.get(int(user_id), ""),
            int(user_id),
            " ".join(rec_items),
            " ".join(rec_books),
        ])

print(f"Outputs saved to: {OUTPUT_DIR}")

Pure CF metrics:
  k: 10.0
  users_evaluated: 79153.0
  precision_at_k: 0.08899220497012116
  recall_at_k: 0.2411814115570039
  ndcg_at_k: 0.20412913492932713
Outputs saved to: outputs/pure_cf_20260524_142945


In [ ]:
# ===== XUẤT KẾT QUẢ SINGLE MODEL RA CSV =====
import pandas as pd
from datetime import datetime

single_model_log_path = "/content/drive/MyDrive/single_model_results_log.csv"

record = {
    "timestamp":       datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "run_dir":         OUTPUT_DIR,
    # Config model
    "loss":            LOSS,
    "no_components":   NO_COMPONENTS,
    "learning_rate":   LEARNING_RATE,
    "epochs":          EPOCHS,
    # Metrics
    "k":               metrics["k"],
    "users_evaluated": metrics["users_evaluated"],
    "precision_at_k":  metrics["precision_at_k"],
    "recall_at_k":     metrics["recall_at_k"],
    "ndcg_at_k":       metrics.get("ndcg_at_k", None),
}

df_new = pd.DataFrame([record])

# Append vào file log (tạo mới nếu chưa có)
if os.path.exists(single_model_log_path):
    df_existing = pd.read_csv(single_model_log_path)
    df_new = pd.concat([df_existing, df_new], ignore_index=True)

df_new.to_csv(single_model_log_path, index=False)

print(f"\n✅ Đã lưu kết quả vào Drive: {single_model_log_path}")
print(df_new.tail(1).to_string(index=False))  # in dòng vừa thêm

# Tải về máy
from google.colab import files
files.download(single_model_log_path)


✅ Đã lưu kết quả vào Drive: /content/drive/MyDrive/single_model_results_log.csv
          timestamp                         run_dir loss  no_components  learning_rate  epochs    k  users_evaluated  precision_at_k  recall_at_k  ndcg_at_k
2026-05-24 14:35:36 outputs/pure_cf_20260524_142945 warp             64           0.05      20 10.0          79153.0        0.088992     0.241181   0.204129


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Hyperparameter search with early stopping (optional)
import itertools
import pickle

param_grid = {
    "no_components": [64, 128],
    "learning_rate": [0.01, 0.05],
    "loss": ["warp", "bpr"],
}

# Balanced preset
MAX_EPOCHS = 50
EVAL_EVERY = 5
PATIENCE = 3
MIN_DELTA = 1e-4
K_EVAL = K
USE_NDCG_SEARCH = USE_NDCG

keys, values = zip(*param_grid.items())
combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
print(f"\nEarly-stopping search with {len(combinations)} configs...")

best_overall = -1.0
best_params = None
best_metrics = None
results = []
num_items = train_interactions_csr.shape[1]
item_ids = np.arange(num_items, dtype=np.int64)

def eval_precision(model):
    def score_fn(user_id: int) -> np.ndarray:
        return model.predict(user_id, item_ids)
    metrics = evaluate_ranking(
        score_fn,
        train_interactions_csr,
        test_interactions,
        k=K_EVAL,
        use_ndcg=USE_NDCG_SEARCH,
    )
    return metrics["precision_at_k"], metrics

for i, params in enumerate(combinations):
    print(f"\n[Config {i+1}/{len(combinations)}] {params}")
    model = LightFM(
        loss=params["loss"],
        no_components=params["no_components"],
        learning_rate=params["learning_rate"],
        random_state=RANDOM_STATE,
    )
    best_score = -1.0
    best_epoch = 0
    best_state = None
    no_improve = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.fit_partial(
            train_interactions,
            sample_weight=train_weights,
            epochs=1,
            num_threads=NUM_THREADS,
        )
        if epoch % EVAL_EVERY != 0:
            continue
        score, metrics = eval_precision(model)
        print(f"  Epoch {epoch}: Precision@{K_EVAL}={score:.6f}")
        if score > best_score + MIN_DELTA:
            best_score = score
            best_epoch = epoch
            best_state = pickle.dumps(model)
            best_metrics = metrics
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= PATIENCE:
            print(f"  Early stop (patience={PATIENCE}) at epoch {epoch}")
            break

    if best_state is not None:
        model = pickle.loads(best_state)
    results.append({
        "params": params,
        "best_precision": best_score,
        "best_epoch": best_epoch,
    })
    if best_score > best_overall:
        best_overall = best_score
        best_params = params
        with open(os.path.join(OUTPUT_DIR, "best_pure_cf_model.pkl"), "wb") as f:
            pickle.dump(model, f)

print("\n" + "="*50)
print("BEST RESULT (Early-Stopping)")
print("="*50)
print(f"Best Precision@{K_EVAL}: {best_overall:.6f}")
print(f"Best params: {best_params}")
print("Best model saved to: best_pure_cf_model.pkl")

import pandas as pd
df_results = pd.DataFrame([
    {**r["params"], "best_precision": r["best_precision"], "best_epoch": r["best_epoch"]}
    for r in results
 ])
print("\nDetailed results:")
print(df_results.sort_values(by="best_precision", ascending=False))


Early-stopping search with 8 configs...

[Config 1/8] {'no_components': 64, 'learning_rate': 0.01, 'loss': 'warp'}
  Epoch 5: Precision@10=0.062915
  Epoch 10: Precision@10=0.066793
  Epoch 15: Precision@10=0.069522
  Epoch 20: Precision@10=0.071022
  Epoch 25: Precision@10=0.072167
  Epoch 30: Precision@10=0.072938
  Epoch 35: Precision@10=0.073604
  Epoch 40: Precision@10=0.074189
  Epoch 45: Precision@10=0.074801
  Epoch 50: Precision@10=0.075306

[Config 2/8] {'no_components': 64, 'learning_rate': 0.01, 'loss': 'bpr'}
  Epoch 5: Precision@10=0.046937
  Epoch 10: Precision@10=0.053194
  Epoch 15: Precision@10=0.054847
  Epoch 20: Precision@10=0.054884
  Epoch 25: Precision@10=0.054937
  Epoch 30: Precision@10=0.055082
  Epoch 35: Precision@10=0.054868
  Epoch 40: Precision@10=0.054752
  Epoch 45: Precision@10=0.054549
  Early stop (patience=3) at epoch 45

[Config 3/8] {'no_components': 64, 'learning_rate': 0.05, 'loss': 'warp'}
  Epoch 5: Precision@10=0.078977
  Epoch 10: Precisio

In [ ]:


# ============================================================
# LOG KẾT QUẢ TỐI ƯU SAU HYPERPARAMETER SEARCH
# ============================================================
import pandas as pd
import os
from datetime import datetime

# --- 1. File 1: Tất cả kết quả của các config đã thử ---
# Mỗi dòng là 1 tổ hợp param với precision tốt nhất đạt được
all_results_path = os.path.join(OUTPUT_DIR, "all_param_results.csv")

df_all = pd.DataFrame([
    {
        # Hyperparameters
        "no_components":  r["params"]["no_components"],
        "learning_rate":  r["params"]["learning_rate"],
        "loss":           r["params"]["loss"],
        # Kết quả tốt nhất của config đó
        "best_precision": r["best_precision"],
        "best_epoch":     r["best_epoch"],
        # Cài đặt search để dễ so sánh về sau
        "max_epochs":     MAX_EPOCHS,
        "eval_every":     EVAL_EVERY,
        "patience":       PATIENCE,
        "k_eval":         K_EVAL,
    }
    for r in results
])

# Sắp xếp từ tốt → kém để dễ đọc
df_all = df_all.sort_values(by="best_precision", ascending=False).reset_index(drop=True)
df_all.to_csv(all_results_path, index=False)
print(f"\n📊 Toàn bộ kết quả search ({len(results)} configs):")
print(df_all.to_string(index=False))

# --- 2. File 2: Chỉ kết quả tốt nhất, kèm đầy đủ metrics ---
# File này dùng kiểu APPEND — mỗi lần chạy thêm 1 dòng mới
# → Rất tiện để so sánh nhiều lần experiment khác nhau
best_log_path = "best_results_log.csv"   # nằm ngoài OUTPUT_DIR để tích lũy qua nhiều lần chạy

best_record = {
    "timestamp":       datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "run_dir":         OUTPUT_DIR,
    # Hyperparameters tối ưu
    "best_no_components": best_params["no_components"],
    "best_learning_rate": best_params["learning_rate"],
    "best_loss":          best_params["loss"],
    "best_epoch":         next(            # lấy best_epoch của config thắng
        r["best_epoch"] for r in results
        if r["params"] == best_params
    ),
    # Metrics đầy đủ tại config tối ưu
    "precision_at_k":  best_metrics.get("precision_at_k", None),
    "recall_at_k":     best_metrics.get("recall_at_k",    None),
    "ndcg_at_k":       best_metrics.get("ndcg_at_k",      None),
    "users_evaluated": best_metrics.get("users_evaluated",None),
    "k":               K_EVAL,
    # Search settings
    "max_epochs":      MAX_EPOCHS,
    "patience":        PATIENCE,
    "num_configs":     len(combinations),
}

df_best_new = pd.DataFrame([best_record])

# Append vào file log tổng (tạo mới nếu chưa có)
if os.path.exists(best_log_path):
    df_best_existing = pd.read_csv(best_log_path)
    df_best_log = pd.concat([df_best_existing, df_best_new], ignore_index=True)
else:
    df_best_log = df_best_new

df_best_log.to_csv(best_log_path, index=False)

print(f"\n🏆 Kết quả tối ưu:")
print(f"   Best Precision@{K_EVAL}: {best_overall:.6f}")
print(f"   Best Recall@{K_EVAL}:    {best_metrics.get('recall_at_k', 'N/A'):.6f}")
print(f"   Best NDCG@{K_EVAL}:      {best_metrics.get('ndcg_at_k', 'N/A'):.6f}")
print(f"   Best params: {best_params}")
print(f"\n✅ Đã lưu vào:")
print(f"   {all_results_path}   ← toàn bộ {len(results)} configs")
print(f"   {best_log_path}   ← log tích lũy qua các lần chạy")

# --- 3. Tải file về máy (chỉ cần trên Google Colab) ---
from google.colab import files
files.download(all_results_path)
files.download(best_log_path)


📊 Toàn bộ kết quả search (8 configs):
 no_components  learning_rate loss  best_precision  best_epoch  max_epochs  eval_every  patience  k_eval
           128           0.05 warp        0.094301          40          50           5         3      10
            64           0.05 warp        0.092295          50          50           5         3      10
           128           0.01 warp        0.078540          50          50           5         3      10
            64           0.01 warp        0.075306          50          50           5         3      10
            64           0.01  bpr        0.055082          30          50           5         3      10
           128           0.01  bpr        0.054217          15          50           5         3      10
            64           0.05  bpr        0.048703           5          50           5         3      10
           128           0.05  bpr        0.046615           5          50           5         3      10

🏆 Kết quả tối ư

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>